| Python | 3.10 |

| mediapipe | 0.10.9 |

| opencv-python | 4.8.0.76 |

| protobuf | 4.25.3 |

| numpy | 1.24.3 |


import 

import mediapipe as mp

print("cv2:", cv2.__version__)

print("mediapipe:", mp.__version__)

print("Python:", sys.version)

In [1]:
pip install pyautogui

  Using cached PyAutoGUI-0.9.54.tar.gz (61 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyprojec

In [ ]:
time.sleep(2)
print(pyautogui.position())


In [ ]:
pip install pyautogui screeninfo

In [ ]:
# 파이오토 gui를 활용한 화면제어

import cv2
import mediapipe as mp
import math

mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

PANEL_HEIGHT   = 60   #패널높이
HOVER_FRAMES   = 15   # 버튼 위에 N프레임 머물면 선택 (오인식 방지)

current_color     = (0, 255, 0)
current_thickness = 5

index_trail = []
all_trails  = []

# 호버 상태 추적
hover_btn      = None   # 현재 커서가 올라간 버튼 정보
hover_count    = 0      # 연속 프레임 수

color_buttons = [
    ("R", (0, 0, 255),   (0, 0, 255)),
    ("G", (0, 255, 0),   (0, 255, 0)),
    ("B", (255, 0, 0),   (255, 0, 0)),
    ("W", (255,255,255), (200,200,200)),
    ("Y", (0, 255, 255), (0, 255, 255)),
]

thickness_buttons = [
    ("THIN",  1),
    ("MID",   5),
    ("THICK", 10),
]


# ── 약지(16번) 올라왔는지 → CLEAR 트리거 ─────────────
def is_ring_finger_up(lm):
    return lm.landmark[16].y < lm.landmark[14].y


# ── 중지(12번) 올라왔는지 → 메뉴 모드 트리거 ─────────
# 중지 끝(12)이 중지 두 번째 마디(10 PIP)보다 위에 있으면 뻗은 것
def is_middle_finger_up(lm):
    return lm.landmark[12].y < lm.landmark[10].y


# ── 패널 그리기 ────────────────────────────────────────
def draw_panel(frame, w, hover_btn, hover_count):
    cv2.rectangle(frame, (0, 0), (w, PANEL_HEIGHT), (40, 40, 40), -1)
    #버튼크기
    BTN_W = 55
    BTN_H = 44
    GAP   = 5
    Y1    = 8
    Y2    = Y1 + BTN_H
    buttons = []

    # 색상 버튼
    for i, (label, color, btn_color) in enumerate(color_buttons):
        x1 = 10 + i * (BTN_W + GAP)
        x2 = x1 + BTN_W

        is_selected = (current_color == color)
        is_hovered  = (hover_btn == ('color', color))

        border = 3 if is_selected else 1
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), btn_color, -1)
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), (255, 255, 255), border)

        
        # 호버 중이면 진행 바 표시
        if is_hovered and hover_count > 0:
            progress = int((BTN_W - 4) * hover_count / HOVER_FRAMES)
            cv2.rectangle(frame, (x1 + 2, Y2 - 5), # 시작점 (버튼 왼쪽 하단 근처)
                          (x1 + 2 + progress, Y2 - 2), # 끝점   (progress만큼 오른쪽으로)
                          (255, 255, 0),# 노란색
                          -1) # 꽉 채우기

        buttons.append((x1, Y1, x2, Y2, 'color', color))

    # 굵기 버튼
    color_end_x = 10 + len(color_buttons) * (BTN_W + GAP) + 10

    for i, (label, thick) in enumerate(thickness_buttons):
        x1 = color_end_x + i * (BTN_W + GAP)
        x2 = x1 + BTN_W

        is_selected = (current_thickness == thick)
        is_hovered  = (hover_btn == ('thickness', thick))

        border = 3 if is_selected else 1
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), (80, 80, 80), -1)
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), (255, 255, 255), border)

        # 호버 진행 바
        if is_hovered and hover_count > 0:
            progress = int((BTN_W - 4) * hover_count / HOVER_FRAMES)
            cv2.rectangle(frame, (x1 + 2, Y2 - 5), (x1 + 2 + progress, Y2 - 2),
                          (255, 255, 0), -1)

        mid_y = (Y1 + Y2) // 2
        cv2.line(frame, (x1 + 8, mid_y), (x2 - 8, mid_y), (255, 255, 255), thick)
        buttons.append((x1, Y1, x2, Y2, 'thickness', thick))

    return buttons


def get_hovered_button(mx, my, buttons):
    for (x1, y1, x2, y2, btn_type, value) in buttons:
        if x1 < mx < x2 and y1 < my < y2:
            return (btn_type, value)
    return None


# ── 창 설정 ───────────────────────────────────────────
cv2.namedWindow("electronic board", cv2.WINDOW_NORMAL)
cv2.resizeWindow("electronic board", 1280, 720)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        h, w, _ = frame.shape

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False
        results = hands.process(frame_rgb)
        frame_rgb.flags.writeable = True
        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        buttons = draw_panel(frame, w, hover_btn, hover_count)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                lm = hand_landmarks

                # 주요 좌표
                ix = int(lm.landmark[8].x * w)    # 검지 끝
                iy = int(lm.landmark[8].y * h)
                mx = int(lm.landmark[12].x * w)   # 중지 끝
                my = int(lm.landmark[12].y * h)
                rx = int(lm.landmark[16].x * w)   # 약지 끝
                ry = int(lm.landmark[16].y * h)

                # ══ 우선순위 1: 약지 올라오면 CLEAR ══════════
                if is_ring_finger_up(lm):
                    if index_trail:
                        all_trails.append((index_trail.copy(),
                                           current_color, current_thickness))
                        index_trail.clear()
                    all_trails.clear()
                    hover_btn   = None
                    hover_count = 0
                    cv2.circle(frame, (rx, ry), 14, (0, 0, 255), -1)
                    cv2.putText(frame, "CLEAR!", (rx + 15, ry),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
                    continue

                # ══ 우선순위 2: 중지 올라오면 메뉴 모드 ══════
                if is_middle_finger_up(lm):

                    # 그리던 선 저장 (일시정지)
                    if index_trail:
                        all_trails.append((index_trail.copy(),
                                           current_color, current_thickness))
                        index_trail.clear()

                    # 중지 커서 표시 (노란색)
                    cv2.circle(frame, (mx, my), 10, (0, 255, 255), -1)
                    cv2.putText(frame, "MENU", (mx + 12, my),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                    # 호버 감지
                    current_hover = get_hovered_button(mx, my, buttons)

                    if current_hover == hover_btn and current_hover is not None:
                        hover_count += 1
                    else:
                        hover_btn   = current_hover
                        hover_count = 0

                    # HOVER_FRAMES 프레임 이상 머물면 선택 확정
                    if hover_count >= HOVER_FRAMES and hover_btn is not None:
                        btn_type, value = hover_btn
                        if btn_type == 'color':
                            current_color = value
                        elif btn_type == 'thickness':
                            current_thickness = value
                        hover_count = 0   # 선택 후 초기화

                    continue  # 그리기 건너뜀

                # ══ 우선순위 3: 일반 → 검지로 그리기 ══════════
                hover_btn   = None
                hover_count = 0

                cv2.circle(frame, (ix, iy), 8, current_color, -1)

                if iy > PANEL_HEIGHT:
                    index_trail.append((ix, iy))


        # 저장된 선 전부 그리기
        for (trail, color, thickness) in all_trails:
            for i in range(1, len(trail)):
                cv2.line(frame, trail[i - 1], trail[i], color, thickness)

        # 현재 그리는 선
        for i in range(1, len(index_trail)):
            cv2.line(frame, index_trail[i - 1], index_trail[i],
                     current_color, current_thickness)

        cv2.imshow("electronic board", frame)

        key = cv2.waitKey(5)
        if key == ord('q'):
            break


cap.release()
cv2.destroyAllWindows()

In [3]:
# 파이오토 gui를 활용한 화면제어

import cv2
import mediapipe as mp
import math
import pyautogui
import screeninfo   # pip install screeninfo

mp_hands = mp.solutions.hands

# pyautogui 안전장치 해제 (빠른 이동을 위해)
pyautogui.FAILSAFE   = True   # 화면 모서리로 이동시 강제 종료 (안전장치 유지)
pyautogui.PAUSE      = 0      # 함수 호출 간 딜레이 제거

# ── 화면 해상도 가져오기 ──────────────────────────────
monitor  = screeninfo.get_monitors()[0]
SCREEN_W = monitor.width    # 예: 1920
SCREEN_H = monitor.height   # 예: 1080

cap = cv2.VideoCapture(0)

PANEL_HEIGHT = 60
HOVER_FRAMES = 15

current_color     = (0, 255, 0)
current_thickness = 5

index_trail = []
all_trails  = []

# 호버 상태 추적
hover_btn   = None
hover_count = 0

# 마우스 제어 상태
prev_mouse_x  = None   # 이전 프레임 마우스 x (스무딩용)
prev_mouse_y  = None   # 이전 프레임 마우스 y
is_clicking   = False  # 현재 클릭 중인지
SMOOTH        = 0.4    # 마우스 이동 스무딩 (0~1, 낮을수록 부드럽고 느림)

# 마우스 제어 영역 (카메라 중앙 일부만 사용 → 정확도 향상)
# 카메라 좌표 0.0~1.0 중에서 이 범위만 마우스 영역으로 매핑
CAM_X_MIN, CAM_X_MAX = 0.1, 0.9
CAM_Y_MIN, CAM_Y_MAX = 0.1, 0.9

color_buttons = [
    ("R", (0, 0, 255),   (0, 0, 255)),
    ("G", (0, 255, 0),   (0, 255, 0)),
    ("B", (255, 0, 0),   (255, 0, 0)),
    ("W", (255,255,255), (200,200,200)),
    ("Y", (0, 255, 255), (0, 255, 255)),
]

thickness_buttons = [
    ("THIN",  1),
    ("MID",   5),
    ("THICK", 10),
]


# ── 손가락 감지 함수들 ────────────────────────────────

def is_ring_finger_up(lm):
    """약지(16번) → CLEAR"""
    return lm.landmark[16].y < lm.landmark[14].y

def is_middle_finger_up(lm):
    """중지(12번) → 메뉴 모드"""
    return lm.landmark[12].y < lm.landmark[10].y

def is_pinky_up(lm):
    """새끼손가락(20번) → 마우스 제어 모드"""
    return lm.landmark[20].y < lm.landmark[18].y

def is_thumb_index_pinch(lm, w, h):
    """
    엄지(4번) + 검지(8번) 거리가 가까우면 → 마우스 클릭
    마우스 모드에서만 사용
    """
    x4 = lm.landmark[4].x * w
    y4 = lm.landmark[4].y * h
    x8 = lm.landmark[8].x * w
    y8 = lm.landmark[8].y * h
    return math.sqrt((x4 - x8) ** 2 + (y4 - y8) ** 2) < 30


def cam_to_screen(norm_x, norm_y):
    """
    카메라 정규화 좌표(0~1) → 실제 화면 픽셀 좌표 변환
    CAM_X_MIN~MAX 범위를 화면 전체에 매핑
    """
    # 범위 클리핑
    cx = max(CAM_X_MIN, min(CAM_X_MAX, norm_x))
    cy = max(CAM_Y_MIN, min(CAM_Y_MAX, norm_y))

    # 0~1로 정규화
    rx = (cx - CAM_X_MIN) / (CAM_X_MAX - CAM_X_MIN)
    ry = (cy - CAM_Y_MIN) / (CAM_Y_MAX - CAM_Y_MIN)

    # 화면 좌우 반전 (카메라는 거울상)
    rx = 1.0 - rx

    sx = int(rx * SCREEN_W)
    sy = int(ry * SCREEN_H)
    return sx, sy


# ── 패널 그리기 ────────────────────────────────────────
def draw_panel(frame, w, hover_btn, hover_count):
    cv2.rectangle(frame, (0, 0), (w, PANEL_HEIGHT), (40, 40, 40), -1)

    BTN_W = 55
    BTN_H = 44
    GAP   = 5
    Y1    = 8
    Y2    = Y1 + BTN_H
    buttons = []

    # 색상 버튼
    for i, (label, color, btn_color) in enumerate(color_buttons):
        x1 = 10 + i * (BTN_W + GAP)
        x2 = x1 + BTN_W

        is_selected = (current_color == color)
        is_hovered  = (hover_btn == ('color', color))

        border = 3 if is_selected else 1
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), btn_color, -1)
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), (255, 255, 255), border)

        if is_hovered and hover_count > 0:
            progress = int((BTN_W - 4) * hover_count / HOVER_FRAMES)
            cv2.rectangle(frame, (x1 + 2, Y2 - 5),
                          (x1 + 2 + progress, Y2 - 2), (255, 255, 0), -1)

        cv2.putText(frame, label, (x1 + 18, Y1 + 29),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)
        buttons.append((x1, Y1, x2, Y2, 'color', color))

    # 굵기 버튼
    color_end_x = 10 + len(color_buttons) * (BTN_W + GAP) + 10

    for i, (label, thick) in enumerate(thickness_buttons):
        x1 = color_end_x + i * (BTN_W + GAP)
        x2 = x1 + BTN_W

        is_selected = (current_thickness == thick)
        is_hovered  = (hover_btn == ('thickness', thick))

        border = 3 if is_selected else 1
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), (80, 80, 80), -1)
        cv2.rectangle(frame, (x1, Y1), (x2, Y2), (255, 255, 255), border)

        if is_hovered and hover_count > 0:
            progress = int((BTN_W - 4) * hover_count / HOVER_FRAMES)
            cv2.rectangle(frame, (x1 + 2, Y2 - 5),
                          (x1 + 2 + progress, Y2 - 2), (255, 255, 0), -1)

        mid_y = (Y1 + Y2) // 2
        cv2.line(frame, (x1 + 8, mid_y), (x2 - 8, mid_y), (255, 255, 255), thick)
        buttons.append((x1, Y1, x2, Y2, 'thickness', thick))

    return buttons


def get_hovered_button(mx, my, buttons):
    for (x1, y1, x2, y2, btn_type, value) in buttons:
        if x1 < mx < x2 and y1 < my < y2:
            return (btn_type, value)
    return None


# ── 창 설정 ───────────────────────────────────────────
cv2.namedWindow("electronic board", cv2.WINDOW_NORMAL)
cv2.resizeWindow("electronic board", 1280, 720)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        h, w, _ = frame.shape

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False
        results = hands.process(frame_rgb)
        frame_rgb.flags.writeable = True
        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        buttons = draw_panel(frame, w, hover_btn, hover_count)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                lm = hand_landmarks

                # 주요 좌표 (카메라 픽셀)
                ix = int(lm.landmark[8].x * w)
                iy = int(lm.landmark[8].y * h)
                mx_cam = int(lm.landmark[12].x * w)
                my_cam = int(lm.landmark[12].y * h)
                rx = int(lm.landmark[16].x * w)
                ry = int(lm.landmark[16].y * h)
                px = int(lm.landmark[20].x * w)   # 새끼 끝
                py = int(lm.landmark[20].y * h)

                # ══ 우선순위 1: 약지 → CLEAR ══════════════════
                if is_ring_finger_up(lm):
                    if index_trail:
                        all_trails.append((index_trail.copy(),
                                           current_color, current_thickness))
                        index_trail.clear()
                    all_trails.clear()
                    hover_btn   = None
                    hover_count = 0
                    cv2.circle(frame, (rx, ry), 14, (0, 0, 255), -1)
                    cv2.putText(frame, "CLEAR!", (rx + 15, ry),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
                    continue

                # ══ 우선순위 2: 새끼손가락 → 마우스 제어 모드 ══
                if is_pinky_up(lm):

                    # 그리던 선 저장
                    if index_trail:
                        all_trails.append((index_trail.copy(),
                                           current_color, current_thickness))
                        index_trail.clear()

                    hover_btn   = None
                    hover_count = 0

                    # 검지 좌표로 마우스 이동 (정규화 좌표 사용)
                    norm_x = lm.landmark[8].x
                    norm_y = lm.landmark[8].y
                    sx, sy = cam_to_screen(norm_x, norm_y)

                    # 스무딩 적용 (이전 위치와 현재 위치 보간)
                    if prev_mouse_x is None:
                        prev_mouse_x, prev_mouse_y = sx, sy

                    smooth_x = int(prev_mouse_x + (sx - prev_mouse_x) * SMOOTH)
                    smooth_y = int(prev_mouse_y + (sy - prev_mouse_y) * SMOOTH)
                    prev_mouse_x, prev_mouse_y = smooth_x, smooth_y

                    pyautogui.moveTo(smooth_x, smooth_y)

                    # 엄지+검지 오므리면 클릭
                    pinch = is_thumb_index_pinch(lm, w, h)

                    if pinch and not is_clicking:
                        pyautogui.mouseDown()
                        is_clicking = True
                    elif not pinch and is_clicking:
                        pyautogui.mouseUp()
                        is_clicking = False

                    # 카메라 화면에 마우스 커서 표시
                    color_cursor = (0, 165, 255)   # 주황색
                    cv2.circle(frame, (ix, iy), 10, color_cursor, -1)
                    cv2.circle(frame, (ix, iy), 14, color_cursor, 2)

                    # 클릭 중이면 빨간 원 추가
                    if is_clicking:
                        cv2.circle(frame, (ix, iy), 20, (0, 0, 255), 3)

                    # 화면 좌표 표시
                    cv2.putText(frame, f"MOUSE ({smooth_x},{smooth_y})",
                                (px + 12, py),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

                    # 카메라 유효 제어 영역 시각화 (반투명 사각형)
                    ax1 = int(CAM_X_MIN * w)
                    ay1 = int(CAM_Y_MIN * h)
                    ax2 = int(CAM_X_MAX * w)
                    ay2 = int(CAM_Y_MAX * h)
                    overlay = frame.copy()
                    cv2.rectangle(overlay, (ax1, ay1), (ax2, ay2), (0, 165, 255), 1)
                    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)

                    continue

                # ══ 우선순위 3: 중지 → 메뉴 모드 ══════════════
                if is_middle_finger_up(lm):

                    if index_trail:
                        all_trails.append((index_trail.copy(),
                                           current_color, current_thickness))
                        index_trail.clear()

                    prev_mouse_x = prev_mouse_y = None
                    is_clicking  = False

                    cv2.circle(frame, (mx_cam, my_cam), 10, (0, 255, 255), -1)
                    cv2.putText(frame, "MENU", (mx_cam + 12, my_cam),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                    current_hover = get_hovered_button(mx_cam, my_cam, buttons)

                    if current_hover == hover_btn and current_hover is not None:
                        hover_count += 1
                    else:
                        hover_btn   = current_hover
                        hover_count = 0

                    if hover_count >= HOVER_FRAMES and hover_btn is not None:
                        btn_type, value = hover_btn
                        if btn_type == 'color':
                            current_color = value
                        elif btn_type == 'thickness':
                            current_thickness = value
                        hover_count = 0

                    continue

                # ══ 우선순위 4: 일반 → 검지로 그리기 ══════════
                hover_btn    = None
                hover_count  = 0
                prev_mouse_x = prev_mouse_y = None
                is_clicking  = False

                cv2.circle(frame, (ix, iy), 8, current_color, -1)

                if iy > PANEL_HEIGHT:
                    index_trail.append((ix, iy))

        else:
            # 손 없음 → 마우스 버튼 혹시 눌려있으면 해제
            if is_clicking:
                pyautogui.mouseUp()
                is_clicking = False
            prev_mouse_x = prev_mouse_y = None
            hover_btn    = None
            hover_count  = 0
            if index_trail:
                all_trails.append((index_trail.copy(),
                                   current_color, current_thickness))
                index_trail.clear()

        # 저장된 선 전부 그리기
        for (trail, color, thickness) in all_trails:
            for i in range(1, len(trail)):
                cv2.line(frame, trail[i - 1], trail[i], color, thickness)

        # 현재 그리는 선
        for i in range(1, len(index_trail)):
            cv2.line(frame, index_trail[i - 1], index_trail[i],
                     current_color, current_thickness)

        cv2.imshow("electronic board", frame)

        key = cv2.waitKey(5)
        if key == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [4]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np

# 1. 초기 설정
cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils 

screen_width, screen_height = pyautogui.size()

while True:
    success, img = cap.read()
    if not success: break

    img = cv2.flip(img, 1)
    h, w, c = img.shape

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS)

            index_finger = hand_lms.landmark[8]
            thumb_finger = hand_lms.landmark[4]

            mouse_x = np.interp(index_finger.x, [0, 1], [0, screen_width])
            mouse_y = np.interp(index_finger.y, [0, 1], [0, screen_height])

            pyautogui.moveTo(mouse_x, mouse_y, _pause=False)

            ix, iy = int(index_finger.x * w), int(index_finger.y * h)
            tx, ty = int(thumb_finger.x * w), int(thumb_finger.y * h)
            
            distance = np.sqrt((ix - tx)**2 + (iy - ty)**2)

            if distance < 30:
                pyautogui.click()
                cv2.putText(img, "CLICK!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Virtual Mouse", img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.

In [5]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time 

cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

screen_width, screen_height = pyautogui.size()

cv2.namedWindow("Virtual Mouse", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Virtual Mouse", 320, 240)

while True:
    success, img = cap.read()
    if not success: break

    img = cv2.flip(img, 1)
    h, w, c = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS)

            index_finger = hand_lms.landmark[8] 
            middle_finger = hand_lms.landmark[12]
            thumb_finger = hand_lms.landmark[4]

            mouse_x = np.interp(index_finger.x, [0, 1], [0, screen_width])
            mouse_y = np.interp(index_finger.y, [0, 1], [0, screen_height])
            pyautogui.moveTo(mouse_x, mouse_y, _pause=False)

            ix, iy = int(index_finger.x * w), int(index_finger.y * h)
            mx, my = int(middle_finger.x * w), int(middle_finger.y * h)
            tx, ty = int(thumb_finger.x * w), int(thumb_finger.y * h)
            
            dist_single = np.sqrt((ix - tx)**2 + (iy - ty)**2)
            dist_double = np.sqrt((mx - tx)**2 + (my - ty)**2)

            if dist_single < 30:
                pyautogui.click()
                cv2.putText(img, "SINGLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                time.sleep(0.3) 

            elif dist_double < 30:
                pyautogui.doubleClick()
                cv2.putText(img, "DOUBLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                time.sleep(0.3) 

    cv2.imshow("Virtual Mouse", img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.

In [6]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time 

cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

screen_width, screen_height = pyautogui.size()

cv2.namedWindow("Virtual Mouse", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Virtual Mouse", 320, 240)

while True:
    success, img = cap.read()
    if not success: break

    img = cv2.flip(img, 1)
    h, w, c = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS)

            index_finger = hand_lms.landmark[8] 
            middle_finger = hand_lms.landmark[12]
            thumb_finger = hand_lms.landmark[4]

            mouse_x = np.interp(index_finger.x, [0, 1], [0, screen_width])
            mouse_y = np.interp(index_finger.y, [0, 1], [0, screen_height])
            pyautogui.moveTo(mouse_x, mouse_y, _pause=False)

            ix, iy = int(index_finger.x * w), int(index_finger.y * h)
            mx, my = int(middle_finger.x * w), int(middle_finger.y * h)
            tx, ty = int(thumb_finger.x * w), int(thumb_finger.y * h)
            
            dist_single = np.sqrt((ix - tx)**2 + (iy - ty)**2)
            dist_double = np.sqrt((mx - tx)**2 + (my - ty)**2)

            if dist_single < 30:
                pyautogui.click()
                cv2.putText(img, "SINGLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                time.sleep(0.3) 

            elif dist_double < 30:
                pyautogui.doubleClick()
                cv2.putText(img, "DOUBLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                time.sleep(0.3) 

    cv2.imshow("Virtual Mouse", img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time
import math

cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

screen_width, screen_height = pyautogui.size()
frame_r = 150 

cv2.namedWindow("Virtual Mouse", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Virtual Mouse", 320, 240) 

plocX, plocY = 0, 0
clocX, clocY = 0, 0
smoothening = 5

while True:
    success, img = cap.read()
    if not success: break

    img = cv2.flip(img, 1)
    h, w, c = img.shape
    
    cv2.rectangle(img, (frame_r, frame_r), (w - frame_r, h - frame_r), (255, 0, 255), 2)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS)

            index_finger = hand_lms.landmark[8]  
            middle_finger = hand_lms.landmark[12] 
            thumb_finger = hand_lms.landmark[4]  

            ix, iy = int(index_finger.x * w), int(index_finger.y * h)
            mx, my = int(middle_finger.x * w), int(middle_finger.y * h)
            tx, ty = int(thumb_finger.x * w), int(thumb_finger.y * h)
            
            mouse_x = np.interp(ix, [frame_r, w - frame_r], [0, screen_width])
            mouse_y = np.interp(iy, [frame_r, h - frame_r], [0, screen_height])
            
            clocX = plocX + (mouse_x - plocX) / smoothening
            clocY = plocY + (mouse_y - plocY) / smoothening

            pyautogui.moveTo(clocX, clocY, _pause=False)

            plocX, plocY = clocX, clocY

            dist_single = math.hypot(ix - tx, iy - ty) 
            dist_double = math.hypot(mx - tx, my - ty) 

            if dist_single < 30:
                pyautogui.click()
                cv2.putText(img, "SINGLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                time.sleep(0.3) 

            elif dist_double < 30:
                pyautogui.doubleClick()
                cv2.putText(img, "DOUBLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                time.sleep(0.3) 

    cv2.imshow("Virtual Mouse", img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.